# Notebook 02 — Data Wrangling & Nettoyage

**Projet :** Prediction du risque d'abandon scolaire  
**Equipe :** Hugo RAGUIN · Amine TALEB · Elliot FIORESE  

Ce notebook applique un **nettoyage rigoureux** des donnees brutes en 5 etapes :

1. Audit de qualite (valeurs manquantes, types, doublons, outliers)
2. Normalisation et uniformisation des formats
3. Imputation des valeurs manquantes
4. Ingenierie de variables pedagogiques
5. Encodage et sauvegarde des tables finales

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath('..')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from student_risk_dataset import ensure_student_dataset
from data_clean import load_raw_data, impute_missing_values, handle_outliers
from utils_viz import set_custom_style

set_custom_style(theme='light')
%matplotlib inline
print('Environnement pret.')

## 1. Chargement du dataset brut

In [ ]:
raw_path = ensure_student_dataset(force=False)
df_raw = load_raw_data(str(raw_path))
print(f'Dataset brut : {df_raw.shape[0]} lignes x {df_raw.shape[1]} colonnes')
df_raw.head(3)

## 2. Audit de qualite

### 2a. Valeurs manquantes

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'n_missing': missing, 'pct_missing': missing_pct})
missing_df = missing_df[missing_df['n_missing'] > 0].sort_values('n_missing', ascending=False)

print(f'Variables avec valeurs manquantes : {len(missing_df)} / {df_raw.shape[1]}')
print(missing_df.to_string())

In [ ]:
# Visualisation des taux de valeurs manquantes
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(missing_df.index, missing_df['pct_missing'], color='#1A73E8', edgecolor='none')
ax.set_xlabel('Taux de valeurs manquantes (%)')
ax.set_title('Audit des valeurs manquantes — Dataset etudiant brut')
ax.bar_label(bars, fmt='%.1f %%', padding=3, fontsize=9)
ax.set_xlim(0, missing_df['pct_missing'].max() * 1.3)
fig.tight_layout()
os.makedirs('report/assets', exist_ok=True)
fig.savefig('report/assets/tp1_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

### 2b. Doublons et plages physiques

In [ ]:
print(f'Doublons : {df_raw.duplicated().sum()}')
print(f'Identifiants uniques : {df_raw["student_id"].nunique()} / {len(df_raw)}')

# Verification des plages physiques admissibles
physical_bounds = {
    'attendance_rate': (0, 100),
    'prior_average': (0, 20),
    'continuous_assessment': (0, 20),
    'stress_index': (0, 100),
    'study_hours_per_week': (0, 168),
    'lms_sessions_week': (0, 168),
}
print('\n=== Verification des plages physiques ===')
for col, (lo, hi) in physical_bounds.items():
    series = df_raw[col].dropna()
    n_out = ((series < lo) | (series > hi)).sum()
    status = 'OK' if n_out == 0 else f'ANOMALIE ({n_out} valeurs)'
    print(f'  {col:30s} [{lo}, {hi}] -> {status}')

In [ ]:
# Utilisation de handle_outliers de data_clean pour tracer les hors-plage
# (demonstration sur attendance_rate — devrait renvoyer 0 anomalie car deja contraint)
df_test = handle_outliers(df_raw.copy(), ['attendance_rate'], min_val=0.0, max_val=100.0)
df_test = handle_outliers(df_test, ['prior_average'], min_val=0.0, max_val=20.0)
print('Verification handle_outliers terminee — aucune anomalie physique detectee dans ce dataset.')

## 3. Nettoyage et uniformisation

**Strategie d'imputation choisie :**
- Variables numeriques : **mediane** (robuste aux valeurs extremes)
- Variables booleennes : **False** (hypothese conservatrice)
- Variables categorielles : **'Unknown'** (tracabilite explicite)

Aucune ligne n'est supprimee : la cohorte complete de 1 600 etudiants est conservee.

In [ ]:
df = df_raw.copy()

# Etape 1 : Normalisation des booleens (gestion des formats mixtes True/False/'True'/'False')
bool_map = {True: True, False: False, 'True': True, 'False': False}
for col in ['scholarship', 'internet_access', 'dropout_risk']:
    if col in df.columns:
        df[col] = df[col].map(bool_map)
print('Etape 1 terminee : booleens normalises.')

In [ ]:
# Etape 2 : Creation de drapeaux de non-reponse pour les variables academiques critiques
# Ces drapeaux permettent au modele d'apprendre si l'absence de donnee est elle-meme un signal
tracked_cols = ['attendance_rate', 'prior_average', 'continuous_assessment']
for col in tracked_cols:
    df[f'{col}_missing'] = df[col].isna().astype(bool)

n_flags = df[[f'{c}_missing' for c in tracked_cols]].sum()
print('Etape 2 terminee : drapeaux de non-reponse crees.')
print(n_flags.to_string())

In [ ]:
# Etape 3 : Imputation numerique par la mediane (via data_clean)
numeric_cols = [
    'age', 'commute_minutes', 'study_hours_per_week', 'lms_sessions_week',
    'attendance_rate', 'assignment_delay_days', 'prior_average',
    'continuous_assessment', 'stress_index'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = impute_missing_values(df, numeric_cols, method='median')
print('Etape 3 terminee : imputation numerique (mediane).')

In [ ]:
# Etape 4 : Imputation booleenne et categorielle
for col in ['scholarship', 'internet_access']:
    df[col] = df[col].fillna(False)
for col in ['program', 'parental_education']:
    df[col] = df[col].fillna('Unknown')

# Validation : zero valeur manquante restante hors cible
remaining = df.drop(columns=['dropout_risk']).isnull().sum().sum()
print(f'Etape 4 terminee. Valeurs manquantes restantes (hors cible) : {remaining}')
print(f'Shape apres nettoyage : {df.shape}')

## 4. Ingenierie de variables pedagogiques

Trois variables composites sont construites pour capturer des dimensions non lineaires du risque :

- **`engagement_score`** : signal global d'implication (assiduite + travail + LMS - retards)
- **`grade_trend_gap`** : tendance de progression (evaluation continue vs. historique)
- **`academic_pressure_index`** : surcharge organisationnelle (stress + retards + absenteisme)

In [ ]:
df['engagement_score'] = (
    df['attendance_rate'] * 0.45
    + df['study_hours_per_week'] * 1.8
    + df['lms_sessions_week'] * 1.5
    - df['assignment_delay_days'] * 3.2
).round(2)

df['grade_trend_gap'] = (df['continuous_assessment'] - df['prior_average']).round(2)

df['academic_pressure_index'] = (
    df['stress_index']
    + df['assignment_delay_days'] * 2.4
    + (100 - df['attendance_rate']) * 0.35
).round(2)

print('Variables derivees creees :')
print(df[['engagement_score', 'grade_trend_gap', 'academic_pressure_index']].describe().round(2))

In [ ]:
# Visualisation : separation des distributions a risque / non a risque
derived_cols = ['engagement_score', 'grade_trend_gap', 'academic_pressure_index']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, derived_cols):
    for val, label, color in [(False, 'Non a risque', '#188038'), (True, 'A risque', '#D93025')]:
        subset = df[df['dropout_risk'] == val][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor='none')
    ax.set_title(col.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('Valeur')
    ax.legend(fontsize=8)

fig.suptitle('Distribution des variables derivees selon le statut de risque', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

## 5. Encodage et sauvegarde

In [ ]:
# Sauvegarde de la table analytique (lisible par les equipes pedagogiques)
os.makedirs('data/processed', exist_ok=True)
df.to_csv('data/processed/tp1_student_risk_wrangled.csv', index=False)
print(f'Table wrangled : {df.shape}  ->  data/processed/tp1_student_risk_wrangled.csv')

In [ ]:
# Encodage One-Hot des variables categorielles + conversion booleens en float
df_model = pd.get_dummies(df, columns=['program', 'parental_education'], drop_first=True)

for col in ['scholarship', 'internet_access', 'dropout_risk',
            'attendance_rate_missing', 'prior_average_missing', 'continuous_assessment_missing']:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype(float)

df_model = df_model.drop(columns=['student_id'], errors='ignore')
df_model.to_csv('data/processed/tp1_student_risk_model_ready.csv', index=False)
print(f'Table model-ready : {df_model.shape}  ->  data/processed/tp1_student_risk_model_ready.csv')

In [ ]:
# Bilan du pipeline de wrangling
print('=' * 55)
print('BILAN DU PIPELINE DE WRANGLING')
print('=' * 55)
print(f'  Dataset brut          : {df_raw.shape[0]} lignes x {df_raw.shape[1]} colonnes')
print(f'  Table analytique      : {df.shape[0]} lignes x {df.shape[1]} colonnes')
print(f'  Table model-ready     : {df_model.shape[0]} lignes x {df_model.shape[1]} colonnes')
print(f'  Lignes supprimees     : 0 (cohorte complete conservee)')
print(f'  Taux de risque        : {df["dropout_risk"].mean()*100:.1f} %')
print(f'  Etudiants a risque    : {int(df["dropout_risk"].sum())} / {len(df)}')
print('=' * 55)